# Identify analyses to rerun

Build the full set of country/analysis combinations that should be used, drop those already usable in `outputs/59597639` or `outputs/59746206`, and write the remainder to `data/config/rerun_pairs.json` for `scripts/massive/jtrauer/rerun_countries.py`.

A combination is in scope if it was requested (`get_analyses_for_country`) and was not skipped (no scaler data). Complete means `store_outputs` finished (`updates.h5` present).

`59597639` is the original full run. `59746206` is the first rerun, which stopped when the remote disk filled. Results from `59597639` are from before the OxCGRT changes (H1 removed; independent also referenced to run start) and the Oceania/Singapore `no_scaling` `beta` initialisation, so those analyses only count as available if they completed in `59746206`. Unchanged analyses are available from either job.

The inventory cell reports how much of each job is actually on disk. If a folder is only a partial copy, combinations that finished remotely but are missing locally will be treated as still needed.


In [1]:
import json
import pandas as pd

from emu_renewal.constants import (
    ANALYSIS_TYPES,
    DATA_PATH,
    OUTPUTS_PATH,
    OXCGRT_ANALYSIS_TYPES,
)
from emu_renewal.run import get_analyses_for_country
from emu_renewal.utils import get_cont_of_country

In [2]:
ORIGINAL_RUN = "59597639"
RERUN_JOB = "59746206"
job_ids = [ORIGINAL_RUN, RERUN_JOB]
countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))


def job_inventory(run_id):
    run_path = OUTPUTS_PATH / run_id
    present = [p.name for p in run_path.iterdir() if p.is_dir()] if run_path.exists() else []
    n_complete = sum(
        (run_path / iso3 / analysis / "updates.h5").exists()
        for iso3 in present
        for analysis in ANALYSIS_TYPES
    )
    return {"countries on disk": len(present), "complete analyses": n_complete}


pd.DataFrame({job: job_inventory(job) for job in job_ids})

,59597639,59746206
countries on disk,33,80
complete analyses,47,229


In [3]:
def classify_analysis(iso3, analysis, run_path, log_text):
    """Match run.py: complete if store_outputs finished, skipped if ScalerException."""
    if (run_path / iso3 / analysis / "updates.h5").exists():
        return "complete"
    if log_text is None:
        return "no log"
    if f"{analysis} data not available" in log_text:
        return "skipped"
    return "not run"


def classify_job(run_id):
    run_path = OUTPUTS_PATH / run_id
    status = pd.DataFrame(index=countries, columns=ANALYSIS_TYPES)
    for iso3 in countries:
        log_path = run_path / iso3 / "run.log"
        log_text = log_path.read_text() if log_path.exists() else None
        requested_types = get_analyses_for_country(iso3)
        for analysis in ANALYSIS_TYPES:
            if analysis not in requested_types:
                status.loc[iso3, analysis] = "not requested"
            else:
                status.loc[iso3, analysis] = classify_analysis(iso3, analysis, run_path, log_text)
    return status


statuses = {job: classify_job(job) for job in job_ids}
pd.concat(
    {job: s.apply(pd.Series.value_counts).fillna(0).astype(int) for job, s in statuses.items()},
    axis=1,
).fillna(0).astype(int)

59597639                                          \
              no_scaling oxcgrt_floored oxcgrt_independent g_mob   
complete              32             12                  2     1   
no log                93             93                 93    91   
not requested          0              0                  0     3   
not run                1             21                 31    31   

                                                 59746206                 \
              fb_visited_mob fb_singletile_mob no_scaling oxcgrt_floored   
complete                   0                 0          1             65   
no log                    91                91         46             46   
not requested              3                 3          0              0   
not run                   32                32         79             15   

                                                                         
              oxcgrt_independent g_mob fb_visited_mob fb_singletile_mob  
complete                      61    26             35                41  
no log                        46    44             44                44  
not requested                  0     3              3                 3  
not run                       19    53             44                38

In [4]:
def pairs_from_mask(mask):
    stacked = mask.stack()
    return list(stacked[stacked].index)


requested = pd.DataFrame(False, index=countries, columns=ANALYSIS_TYPES)
for iso3 in countries:
    requested.loc[iso3, get_analyses_for_country(iso3)] = True

changed = pd.DataFrame(False, index=countries, columns=ANALYSIS_TYPES)
changed[OXCGRT_ANALYSIS_TYPES] = True
oceania = [iso3 for iso3 in countries if get_cont_of_country(iso3) == "OC"]
changed.loc[oceania, "no_scaling"] = True

skipped = (statuses[ORIGINAL_RUN] == "skipped") | (statuses[RERUN_JOB] == "skipped")
complete_original = statuses[ORIGINAL_RUN] == "complete"
complete_rerun = statuses[RERUN_JOB] == "complete"
available = complete_rerun | (~changed & complete_original)
need = requested & ~skipped & ~available
rerun_pairs = sorted(pairs_from_mask(need))

pd.Series(
    {
        "requested": int(requested.to_numpy().sum()),
        "skipped": int((requested & skipped).to_numpy().sum()),
        "available from rerun": int((requested & complete_rerun).to_numpy().sum()),
        "available from original only": int(
            (requested & ~changed & complete_original & ~complete_rerun).to_numpy().sum()
        ),
        "remaining": len(rerun_pairs),
    }
)

requested                       747
skipped                           0
available from rerun            229
available from original only     32
remaining                       486
dtype: int64

In [5]:
pd.Series([analysis for _, analysis in rerun_pairs]).value_counts()

g_mob                 96
no_scaling            94
fb_visited_mob        88
fb_singletile_mob     82
oxcgrt_independent    65
oxcgrt_floored        61
Name: count, dtype: int64

In [6]:
json.dump(rerun_pairs, open(DATA_PATH / "config/rerun_pairs.json", "w"))